In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import average_precision_score
from sklearn.model_selection import StratifiedKFold
from pyod.models.deep_svdd import DeepSVDD
import sys
import scrapbook as sb

sys.path.append('..')
from utils import reshape_to_numpy, interpolate_missing_values, denoise_data, compute_spectrograms, time_avg_pooling

In [2]:
seed = 1
sampling_rate = 10
nperseg=64
noverlap=32
accel_cutoff = 0.4
accel_order = 30
gyro_cutoff = 0.6
gyro_order = 20
batch_size = 32
epochs = 100
hidden_neurons = [128, 64, 32]
n_splits = 5

In [3]:
# Parameters
seed = 18


In [4]:
rng = np.random.RandomState(seed)

In [5]:
data = pd.read_parquet("../data/GBG500.parquet")
data

,ride_id,time_index,ax,ay,az,rx,ry,rz
0,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,0,-2.512796,-9.385012,-1.053078,-0.009155,0.009155,-0.201416
1,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,100,-2.491268,-9.385012,-1.079390,-0.036621,0.027465,-0.155639
2,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,200,-2.534324,-9.382022,-1.030952,0.036621,0.027465,-0.073242
3,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,300,-2.488278,-9.383218,-1.091948,-0.045776,0.036621,-0.183105
4,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,400,-2.483494,-9.382022,-1.100320,-0.045776,0.027465,-0.192260
...,...,...,...,...,...,...,...,...
1529542,ffb6f2f5dc6ba768d88dc0bc1af4cad235526aa9e24168...,241300,0.459862,-9.140430,-3.318900,1.556396,-0.091552,0.274658
1529543,ffb6f2f5dc6ba768d88dc0bc1af4cad235526aa9e24168...,241400,0.455676,-9.140430,-3.321292,1.583861,-0.119018,0.274658
1529544,ffb6f2f5dc6ba768d88dc0bc1af4cad235526aa9e24168...,241500,0.459862,-9.139832,-3.317704,1.583861,-0.109863,0.274658
1529545,ffb6f2f5dc6ba768d88dc0bc1af4cad235526aa9e24168...,241600,0.456274,-9.142224,-3.321292,1.574707,-0.137329,0.283813


In [6]:
labels = pd.read_csv("../data/GBG500_labels.csv")
labels.columns = labels.columns.str.lower()
ride_order_df = pd.DataFrame({"ride_id": data["ride_id"].unique()})
labels_sorted = ride_order_df.merge(
    labels,
    on="ride_id",
    how="left"
)
labels_sorted

,ride_id,label
0,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,Safe
1,00d223cb7aecc9c0cc5871e32a6c45027a068eba07c572...,Reckless
2,02288a4aeca044203394e982e76b87818021dea2b34df9...,Safe
3,0236bdcf13d473ea24d97f5eaeea459f257dffac005694...,Bad weather
4,0238e5dd85143f1b6c59b200bc32f14361b8fc84c51a87...,Safe
...,...,...
495,fe5d7f84d692bbccad8bd566a3ad5c3108fa9f90acd671...,Safe
496,fe67169632306d4668b2affedef510df2cb43e2fb41220...,Safe
497,fee44667fdc8ab4995c702fc3bd36178de0b1ca2c64687...,Safe
498,ff79b2e945b93c2a5efc3a06365267b3a160e7849ba57d...,Safe


In [7]:
data_np = reshape_to_numpy(
    data,
    features = ["ax", "ay", "az", "rx", "ry", "rz"],
    max_timestamps = 4800
)

data_np_clean = interpolate_missing_values(
    data_np,
    method='linear',
    limit=None
)

data_np_clean = denoise_data(
    data=data_np_clean,
    accel_indices=[0, 1, 2],
    gyro_indices=[3, 4, 5],
    accel_cutoff=accel_cutoff,
    accel_order=accel_order,
    gyro_cutoff=gyro_cutoff,
    gyro_order=gyro_order,
)

In [8]:
# Compute spectrograms for all rides
spectrograms_array = compute_spectrograms(data_np_clean, sampling_rate, nperseg, noverlap)
spectrograms_array.shape

(500, 149, 6, 33)

In [9]:
# Aggregate spectrograms using time-averaged pooling
X_feat = time_avg_pooling(spectrograms_array)
X_feat.shape

(500, 198)

In [10]:
# 5-fold stratified CV with Deep SVDD
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=rng.randint(1000))
y_true = (labels_sorted['label'] == 'Reckless').astype(int).values
anomaly_scores = np.full(len(y_true), np.nan)

for fold, (train_idx, test_idx) in enumerate(skf.split(X_feat, y_true)):
    sc = StandardScaler()
    X_train = sc.fit_transform(X_feat[train_idx])
    X_test = sc.transform(X_feat[test_idx])

    np.random.seed(rng.randint(1000))
    model = DeepSVDD(
        n_features=X_train.shape[1],
        hidden_neurons=hidden_neurons,
        epochs=epochs,
        batch_size=batch_size,
        random_state=rng.randint(1000),
    )
    model.fit(X_train)
    anomaly_scores[test_idx] = model.decision_function(X_test)
    print(f"Fold {fold+1}/{n_splits} done")

Epoch 1/100, Loss: 1.692725345492363
Epoch 2/100, Loss: 1.9669364467263222
Epoch 3/100, Loss: 1.9197255373001099
Epoch 4/100, Loss: 1.83103908598423
Epoch 5/100, Loss: 1.9040411449968815
Epoch 6/100, Loss: 1.8124796971678734
Epoch 7/100, Loss: 1.717642068862915
Epoch 8/100, Loss: 1.7601259611546993
Epoch 9/100, Loss: 1.8498362302780151
Epoch 10/100, Loss: 1.7338418439030647
Epoch 11/100, Loss: 1.7632333785295486
Epoch 12/100, Loss: 1.8648235648870468
Epoch 13/100, Loss: 1.6694864332675934
Epoch 14/100, Loss: 1.7675227001309395
Epoch 15/100, Loss: 1.8617753759026527
Epoch 16/100, Loss: 1.6875642463564873


Epoch 17/100, Loss: 1.9315064921975136
Epoch 18/100, Loss: 1.785671778023243
Epoch 19/100, Loss: 1.735267847776413
Epoch 20/100, Loss: 1.7336498126387596
Epoch 21/100, Loss: 1.7880814373493195
Epoch 22/100, Loss: 1.9616932943463326
Epoch 23/100, Loss: 1.8707548156380653
Epoch 24/100, Loss: 1.7820031866431236
Epoch 25/100, Loss: 1.7179906144738197
Epoch 26/100, Loss: 2.0922293812036514
Epoch 27/100, Loss: 1.8126768320798874
Epoch 28/100, Loss: 1.7987233735620975
Epoch 29/100, Loss: 1.9646203443408012
Epoch 30/100, Loss: 1.7384355962276459
Epoch 31/100, Loss: 1.878514289855957


Epoch 32/100, Loss: 1.885160431265831
Epoch 33/100, Loss: 1.839644767343998
Epoch 34/100, Loss: 1.7938988395035267
Epoch 35/100, Loss: 1.9747425019741058
Epoch 36/100, Loss: 1.6581925749778748
Epoch 37/100, Loss: 1.677140273153782
Epoch 38/100, Loss: 1.9331705793738365
Epoch 39/100, Loss: 2.0151083692908287
Epoch 40/100, Loss: 1.8422158807516098
Epoch 41/100, Loss: 1.8164277374744415
Epoch 42/100, Loss: 1.866200402379036
Epoch 43/100, Loss: 1.9105495363473892
Epoch 44/100, Loss: 2.0434958934783936
Epoch 45/100, Loss: 1.800289049744606
Epoch 46/100, Loss: 1.7771769165992737


Epoch 47/100, Loss: 1.785683996975422
Epoch 48/100, Loss: 1.8109031841158867
Epoch 49/100, Loss: 1.8745597898960114
Epoch 50/100, Loss: 1.8677250072360039
Epoch 51/100, Loss: 1.8742422014474869
Epoch 52/100, Loss: 1.9063385054469109
Epoch 53/100, Loss: 1.8262516111135483
Epoch 54/100, Loss: 1.9586573727428913
Epoch 55/100, Loss: 1.9341682270169258
Epoch 56/100, Loss: 1.792044848203659
Epoch 57/100, Loss: 1.8129101917147636
Epoch 58/100, Loss: 1.8498599380254745
Epoch 59/100, Loss: 1.7518300265073776
Epoch 60/100, Loss: 1.8901315331459045
Epoch 61/100, Loss: 1.7703371793031693


Epoch 62/100, Loss: 1.8360568769276142
Epoch 63/100, Loss: 1.9011006727814674
Epoch 64/100, Loss: 1.753777876496315
Epoch 65/100, Loss: 1.8559686094522476
Epoch 66/100, Loss: 1.715373434126377
Epoch 67/100, Loss: 1.7239386141300201
Epoch 68/100, Loss: 2.024063415825367
Epoch 69/100, Loss: 1.842509388923645
Epoch 70/100, Loss: 1.9768280312418938
Epoch 71/100, Loss: 1.802429810166359
Epoch 72/100, Loss: 1.8402618765830994
Epoch 73/100, Loss: 1.7036347463726997
Epoch 74/100, Loss: 1.882582649588585
Epoch 75/100, Loss: 1.8327658027410507
Epoch 76/100, Loss: 1.8648527711629868


...

Epoch 46/100, Loss: 1.8370799347758293
Epoch 47/100, Loss: 1.9169185683131218
Epoch 48/100, Loss: 1.8481809347867966
Epoch 49/100, Loss: 2.4083909392356873
Epoch 50/100, Loss: 1.8222885057330132
Epoch 51/100, Loss: 1.964023657143116
Epoch 52/100, Loss: 1.830587387084961
Epoch 53/100, Loss: 1.985455073416233
Epoch 54/100, Loss: 2.001792810857296
Epoch 55/100, Loss: 2.122680865228176
Epoch 56/100, Loss: 1.8583987951278687
Epoch 57/100, Loss: 1.8526252508163452
Epoch 58/100, Loss: 1.9959837421774864
Epoch 59/100, Loss: 1.9254021868109703
Epoch 60/100, Loss: 2.00465901941061


Epoch 61/100, Loss: 1.8986409902572632
Epoch 62/100, Loss: 1.8861589953303337
Epoch 63/100, Loss: 2.3944770619273186
Epoch 64/100, Loss: 1.9790023565292358
Epoch 65/100, Loss: 1.8934567980468273
Epoch 66/100, Loss: 2.0494469627738
Epoch 67/100, Loss: 1.7573874853551388
Epoch 68/100, Loss: 1.96956317871809
Epoch 69/100, Loss: 1.9297779873013496
Epoch 70/100, Loss: 1.9553316831588745
Epoch 71/100, Loss: 1.9287704899907112
Epoch 72/100, Loss: 1.9916669055819511
Epoch 73/100, Loss: 2.1905850991606712
Epoch 74/100, Loss: 1.8894234597682953
Epoch 75/100, Loss: 1.846837617456913
Epoch 76/100, Loss: 1.970505565404892


Epoch 77/100, Loss: 2.009673148393631
Epoch 78/100, Loss: 1.9786780774593353
Epoch 79/100, Loss: 2.004308633506298
Epoch 80/100, Loss: 2.268220268189907
Epoch 81/100, Loss: 1.902095027267933
Epoch 82/100, Loss: 1.8331909254193306
Epoch 83/100, Loss: 1.9257634207606316
Epoch 84/100, Loss: 1.98174349963665
Epoch 85/100, Loss: 1.820245884358883
Epoch 86/100, Loss: 1.8534337282180786


Epoch 87/100, Loss: 1.8432433120906353
Epoch 88/100, Loss: 2.0190740451216698
Epoch 89/100, Loss: 1.8779736831784248
Epoch 90/100, Loss: 1.8347297087311745
Epoch 91/100, Loss: 1.908999279141426
Epoch 92/100, Loss: 2.021515592932701
Epoch 93/100, Loss: 2.047103226184845
Epoch 94/100, Loss: 1.8811439611017704
Epoch 95/100, Loss: 1.9998201951384544
Epoch 96/100, Loss: 2.0452053397893906


Epoch 97/100, Loss: 1.9229722768068314
Epoch 98/100, Loss: 1.9835537672042847
Epoch 99/100, Loss: 1.9130484387278557
Epoch 100/100, Loss: 2.075090378522873
Fold 5/5 done


In [11]:
ap = average_precision_score(y_true, anomaly_scores)
print(f"Deep SVDD AP (5-fold CV) = {ap:.4f}")
sb.glue("GBG500_ap_spectral_deep_svdd", float(ap))

Deep SVDD AP (5-fold CV) = 0.4982
